# AXIVEST — Portfolio Risk & Options Intelligence
### *“See the risk. Test the downside.”*

**Author & Developer:** Kandala Guru Balaji  
**Program:** B.Tech Computer Science and Business Systems (3rd Year)  
**Project Track:** Financial Data Analytics & Quantitative Risk Modeling  

---

### Project Overview
AXIVEST is an institutional-grade financial analytics and portfolio risk assessment system built on Indian market equities and the NIFTY 50 benchmark. This notebook automates:
1. **Market Data Retrieval & Cleaning:** Historical adjusted close prices via `yfinance` for 10 leading Indian blue-chips and NIFTY 50.
2. **Historical Risk Metrics:** Sample-adjusted annualized volatility ($\sqrt{252}$), peak-to-trough Maximum Drawdown, and aligned Pearson correlation matrices.
3. **Macro Stress Testing:** Scenario shock modeling (Mild, Moderate, Severe) across equal-weighted portfolio holdings (₹100,000 illustrative capital).
4. **Derivatives & Options Intelligence:** Multi-leg option payoff architectures at expiration (Bull Call Spread, Long Straddle, Iron Condor).
5. **Data Export & Database Packaging:** Generation of structured CSVs and automated SQLite database (`axivest.db`) generation for Power BI and Excel.

> **Educational & Analytical Disclaimer:** This project is developed strictly for educational, academic, and portfolio demonstration purposes. All stress testing scenarios and options modeling are illustrative mathematical models and do not constitute financial advice, investment recommendations, or guaranteed return forecasts.

In [ ]:
# Step 1: Install & Import Dependencies
!pip install yfinance

import pandas as pd
import numpy as np
import yfinance as yf
import sqlite3
import os

print('Libraries successfully imported.')

In [ ]:
# Step 2: Define Asset Universe & Configuration
TICKER_MAP = {
    'BHARTIARTL.NS': 'Bharti Airtel',
    'HDFCBANK.NS': 'HDFC Bank',
    'ICICIBANK.NS': 'ICICI Bank',
    'INFY.NS': 'Infosys',
    'ITC.NS': 'ITC',
    'LT.NS': 'L&T',
    'M&M.NS': 'M&M',
    'NTPC.NS': 'NTPC',
    'RELIANCE.NS': 'Reliance',
    'TCS.NS': 'TCS',
    '^NSEI': 'NIFTY 50'
}

TICKERS = list(TICKER_MAP.keys())
TRADING_DAYS = 252
START_DATE = '2024-10-09'
END_DATE = '2026-10-10'
PORTFOLIO_VALUE = 100000.0

print(f'Configured {len(TICKERS)} assets from {START_DATE} to {END_DATE}.')

In [ ]:
# Step 3: Fetch Adjusted Market Prices
raw_data = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=False)

adj_close = raw_data['Adj Close'].copy()
adj_close = adj_close.rename(columns=TICKER_MAP)
adj_close = adj_close.dropna(how='all').ffill().bfill()
adj_close.index = adj_close.index.strftime('%Y-%m-%d')
adj_close.index.name = 'Date'
daily_prices = adj_close.reset_index()

print('Daily Prices Shape:', daily_prices.shape)
daily_prices.head()

In [ ]:
# Step 4: Calculate Daily Returns & Risk Metrics
asset_cols = [c for c in daily_prices.columns if c != 'Date']

daily_returns = daily_prices.copy()
daily_returns[asset_cols] = daily_prices[asset_cols].pct_change()
daily_returns = daily_returns.dropna().reset_index(drop=True)

# Annualized Volatility: Sample Std Dev * sqrt(252) * 100%
vol_records = []
for col in asset_cols:
    ann_vol = daily_returns[col].std(ddof=1) * np.sqrt(TRADING_DAYS) * 100.0
    vol_records.append({
        'stock': col,
        'annualized_volatility_pct': round(ann_vol, 4),
        'asset_type': 'Index' if col == 'NIFTY 50' else 'Equity'
    })
annualized_volatility = pd.DataFrame(vol_records).sort_values('annualized_volatility_pct', ascending=False)

# Maximum Drawdown: Peak-to-Trough Decline
mdd_records = []
for col in asset_cols:
    s = daily_prices[['Date', col]].copy()
    s['peak'] = s[col].cummax()
    s['dd'] = (s[col] - s['peak']) / s['peak'] * 100.0
    mdd_val = s['dd'].min()
    trough_idx = s['dd'].idxmin()
    trough_date = s.loc[trough_idx, 'Date']
    trough_price = s.loc[trough_idx, col]
    peak_price = s.loc[trough_idx, 'peak']
    peak_date = s[s[col] == peak_price]['Date'].iloc[0]
    mdd_records.append({
        'stock': col,
        'max_drawdown_pct': round(mdd_val, 2),
        'peak_date': peak_date,
        'peak_price': round(peak_price, 2),
        'trough_date': trough_date,
        'trough_price': round(trough_price, 2),
        'asset_type': 'Index' if col == 'NIFTY 50' else 'Equity'
    })
maximum_drawdown = pd.DataFrame(mdd_records).sort_values('max_drawdown_pct')

print('Risk Metrics Calculated.')
display(annualized_volatility.head())
display(maximum_drawdown.head())

In [ ]:
# Step 5: Correlation Matrices (Pairwise & Symmetric Grid)
corr_matrix = daily_returns[asset_cols].corr().round(4)

pairwise = []
for i in range(len(asset_cols)):
    for j in range(i + 1, len(asset_cols)):
        pairwise.append({
            'stock_1': asset_cols[i],
            'stock_2': asset_cols[j],
            'correlation': corr_matrix.loc[asset_cols[i], asset_cols[j]]
        })
stock_correlations_pairwise = pd.DataFrame(pairwise)

sym_records = []
for s1 in asset_cols:
    for s2 in asset_cols:
        sym_records.append({
            'row_stock': s1,
            'col_stock': s2,
            'correlation': corr_matrix.loc[s1, s2]
        })
stock_correlations_matrix = pd.DataFrame(sym_records)

print(f'Correlations Generated: {len(stock_correlations_pairwise)} unique pairs, {len(stock_correlations_matrix)} matrix cells.')

In [ ]:
# Step 6: Power BI Unpivoted Prices & Base 100 Rebased Index
unpivoted = pd.melt(daily_prices, id_vars=['Date'], var_name='Stock', value_name='Price')
unpivoted['Date'] = pd.to_datetime(unpivoted['Date'])
unpivoted = unpivoted.sort_values(['Stock', 'Date']).reset_index(drop=True)
unpivoted['Base_Price'] = unpivoted.groupby('Stock')['Price'].transform('first')
unpivoted['Normalized_Price_100'] = (unpivoted['Price'] / unpivoted['Base_Price']) * 100.0
unpivoted['Daily_Return_Pct'] = unpivoted.groupby('Stock')['Price'].pct_change() * 100.0
unpivoted['Date'] = unpivoted['Date'].dt.strftime('%Y-%m-%d')
daily_prices_unpivoted = unpivoted[['Date', 'Stock', 'Price', 'Normalized_Price_100', 'Daily_Return_Pct']]

print('Unpivoted Price History Shape:', daily_prices_unpivoted.shape)
daily_prices_unpivoted.head()

In [ ]:
# Step 7: Macro Stress Testing Simulation
scenarios = {
    'Mild Stress': {
        'TCS': -5.0, 'Infosys': -5.0, 'HDFC Bank': -4.0, 'ICICI Bank': -4.0,
        'Reliance': -5.0, 'NTPC': -3.0, 'L&T': -5.0, 'M&M': -6.0,
        'ITC': -3.0, 'Bharti Airtel': -4.0
    },
    'Moderate Stress': {
        'TCS': -12.0, 'Infosys': -12.0, 'HDFC Bank': -10.0, 'ICICI Bank': -10.0,
        'Reliance': -12.0, 'NTPC': -8.0, 'L&T': -12.0, 'M&M': -15.0,
        'ITC': -7.0, 'Bharti Airtel': -9.0
    },
    'Severe Stress': {
        'TCS': -25.0, 'Infosys': -25.0, 'HDFC Bank': -22.0, 'ICICI Bank': -22.0,
        'Reliance': -25.0, 'NTPC': -18.0, 'L&T': -25.0, 'M&M': -30.0,
        'ITC': -15.0, 'Bharti Airtel': -20.0
    }
}

by_stock = []
summary = []
weight = 10.0

for sc_name, shocks in scenarios.items():
    total_ret = 0.0
    for stock, shock in shocks.items():
        w_impact = shock * (weight / 100.0)
        total_ret += w_impact
        by_stock.append({
            'scenario': sc_name,
            'stock': stock,
            'assumed_shock_pct': shock,
            'portfolio_weight_pct': weight,
            'weighted_impact_pct': round(w_impact, 2)
        })
    port_loss = abs(round(total_ret, 2))
    post_val = round(PORTFOLIO_VALUE * (1 + total_ret / 100.0), 2)
    summary.append({
        'scenario': sc_name,
        'portfolio_return_pct': round(total_ret, 2),
        'portfolio_loss_pct': port_loss,
        'starting_portfolio_value': PORTFOLIO_VALUE,
        'estimated_value_after_stress': post_val
    })

stress_by_stock = pd.DataFrame(by_stock)
stress_summary = pd.DataFrame(summary)
display(stress_summary)

In [ ]:
# Step 8: Options Strategy Expiration Payoff Modeling
strike_range = np.arange(700, 1325, 25)
payoffs = []

for S in strike_range:
    # Bull Call Spread (1000C @ 50, 1100C @ 15 -> Net Debit 35)
    bcs = max(S - 1000, 0) - max(S - 1100, 0) - 35
    payoffs.append({'strategy': 'Bull Call Spread', 'underlying_at_expiry': float(S), 'net_profit_per_unit': float(bcs)})
    
    # Long Straddle (1000C @ 55, 1000P @ 50 -> Net Debit 105)
    ls = max(S - 1000, 0) + max(1000 - S, 0) - 105
    payoffs.append({'strategy': 'Long Straddle', 'underlying_at_expiry': float(S), 'net_profit_per_unit': float(ls)})
    
    # Iron Condor (850P @ 5, 900P @ 14, 1100C @ 18, 1150C @ 4 -> Net Credit 23)
    ic = max(850 - S, 0) - max(900 - S, 0) - max(S - 1100, 0) + max(S - 1150, 0) + 23
    payoffs.append({'strategy': 'Iron Condor', 'underlying_at_expiry': float(S), 'net_profit_per_unit': float(ic)})

options_payoff = pd.DataFrame(payoffs)

options_summary = pd.DataFrame([
    {'strategy': 'Bull Call Spread', 'description': 'Limited-risk bullish strategy (Long 1000C / Short 1100C)', 'net_premium_paid_per_unit': 35.0, 'best_profit_in_tested_range': 65.0, 'worst_profit_in_tested_range': -35.0, 'tested_price_min': 700.0, 'tested_price_max': 1300.0, 'breakeven_lower': 1035.0, 'breakeven_upper': 1035.0},
    {'strategy': 'Long Straddle', 'description': 'Volatility breakout strategy (Long 1000C / Long 1000P)', 'net_premium_paid_per_unit': 105.0, 'best_profit_in_tested_range': 195.0, 'worst_profit_in_tested_range': -105.0, 'tested_price_min': 700.0, 'tested_price_max': 1300.0, 'breakeven_lower': 895.0, 'breakeven_upper': 1105.0},
    {'strategy': 'Iron Condor', 'description': 'Range-bound credit strategy (850P/900P credit spread + 1100C/1150C credit spread)', 'net_premium_paid_per_unit': -23.0, 'best_profit_in_tested_range': 23.0, 'worst_profit_in_tested_range': -27.0, 'tested_price_min': 700.0, 'tested_price_max': 1300.0, 'breakeven_lower': 877.0, 'breakeven_upper': 1123.0}
])

display(options_summary)

In [ ]:
# Step 9: Save Datasets to SQLite Database (axivest.db) & Clean CSVs
conn = sqlite3.connect('axivest.db')
tables = {
    'daily_prices': daily_prices,
    'daily_prices_unpivoted': daily_prices_unpivoted,
    'daily_returns': daily_returns,
    'annualized_volatility': annualized_volatility,
    'maximum_drawdown': maximum_drawdown,
    'stock_correlations_pairwise': stock_correlations_pairwise,
    'stock_correlations_matrix': stock_correlations_matrix,
    'stress_by_stock': stress_by_stock,
    'stress_summary': stress_summary,
    'options_payoff': options_payoff,
    'options_summary': options_summary
}

for name, df in tables.items():
    df.to_sql(name, conn, if_exists='replace', index=False)
    df.to_csv(f'{name}.csv', index=False)
    print(f'Exported: {name}.csv & SQLite Table: {name}')

conn.commit()
conn.close()
print('\naxivest.db successfully populated and verified!')

In [ ]:
# Step 10: Download Artifacts for Power BI Desktop & Excel
try:
    from google.colab import files
    for name in tables.keys():
        files.download(f'{name}.csv')
    files.download('axivest.db')
    print('Download triggered for all project artifacts.')
except Exception as e:
    print('Local environment detected, files saved in local directory:', e)